# Persona Consistency

This notebook computes the four longitudinal persona-consistency metrics used to validate routine/persona continuity on the existing 1K San Francisco no-storm outputs. It also inspects high- and low-scoring agents through their logged stories so the metric behavior can be checked against traces.


## Metric Definitions

The core metrics are:

1. **Role-anchor adherence**: for workers and students, the fraction of weekdays with a substantial primary-role activity block (`Work` for workers, `Education` for students). Homemakers are excluded from this metric because they do not have a fixed external role anchor in the current profile schema.
2. **Anchor start-time stability**: for workers and students with at least three observed weekday anchors, the within-agent standard deviation of first daily `Work`/`Education` start time. Lower is more stable.
3. **Weekday routine continuity**: for each agent, the mean normalized edit-distance similarity across all Monday-Friday compressed activity-chain pairs.
4. **Weekday routine similarity delta**: each agent's weekday routine continuity minus a matched different-agent control from the same model run, role, and weekday. This checks whether the same agent is more self-consistent than generic same-role agents.

For the routine metrics, adjacent duplicate activity states are compressed first. This prevents repeated “continue Home” or “continue Work” decisions from dominating the chain shape.

Let raw activity chain `x` be converted to compressed chain `C(x)` by removing adjacent duplicates. For two compressed chains, similarity is:

$$
\operatorname{sim}(x,y) = 1 - \frac{\operatorname{EditDistance}(C(x), C(y))}{\max(|C(x)|, |C(y)|)}
$$

This gives `1.0` for identical compressed chains. The value decreases as insertions, deletions, or substitutions are needed to align the chains.

For agent `i` with weekday chains `x_{i,d}`, weekday routine continuity is the mean similarity over all weekday pairs:

$$
\operatorname{WRC}_i = \frac{1}{|P_i|}\sum_{(d,d') \in P_i} \operatorname{sim}(x_{i,d}, x_{i,d'})
$$

where `P_i` is the set of Monday-Friday day pairs available for that agent.

The matched-control score compares each agent-day chain to sampled chains from different agents with the same model, role, and weekday. The delta is:

$$
\Delta_i = \operatorname{WRC}_i - \operatorname{ControlSim}_i
$$

A positive delta means the agent is more similar to itself across weekdays than to comparable same-role agents.


In [1]:
from __future__ import annotations

from pathlib import Path
import sys

import pandas as pd
from IPython.display import Markdown, display

ROOT = Path.cwd().resolve()
if ROOT.name == "analysis":
    ROOT = ROOT.parent
ANALYSIS_DIR = ROOT / "analysis"
if str(ANALYSIS_DIR) not in sys.path:
    sys.path.insert(0, str(ANALYSIS_DIR))
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from evals_persona_consistency_ablations import (
    DEFAULT_INPUT_FOLDERS,
    DEFAULT_MODEL_RUNS,
    METRIC_DEFINITIONS,
    evaluate_models,
    get_agent_weekday_chains,
    select_representative_agents,
)
from agent_story import show_agent_story_compare

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 160)

SEED = 42
ANCHOR_MIN_MINUTES = 120
CONTROL_SAMPLES_PER_AGENT_DAY = 20


## Existing Runs

The analysis uses the four existing no-storm San Francisco 1K outputs. These are the same model families used in the current paper evaluation: prompted 20B, prompted 120B, 20B fine-tuned on San Francisco-style supervision, and 20B fine-tuned on broader U.S. supervision.


In [2]:
MODEL_RUNS = DEFAULT_MODEL_RUNS
INPUT_FOLDERS = DEFAULT_INPUT_FOLDERS

pd.DataFrame(
    [
        {"model": model, "run_folder": run, "input_folder": INPUT_FOLDERS[model]}
        for model, run in MODEL_RUNS.items()
    ]
)


,model,run_folder,input_folder
0,OSS_20B,simulation_outputs/SF_1k_oss_20b,Inputs/SF_agents_1K
1,OSS_120B,simulation_outputs/SF_1k_oss_120b,Inputs/SF_agents_1K
2,OSS_20B_FT_SF,simulation_outputs/SF_1k_oss_20b_lora-sf-5epoch,Inputs/SF_agents_1K
3,OSS_20B_FT_US,simulation_outputs/SF_1k_oss_20b_lora-5epoch,Inputs/SF_agents_1K


## Run Metrics

This cell computes all four metrics and displays the model-level summary. The role-anchor threshold is intentionally conservative (`>=120` minutes) so a tiny incidental Work/Education visit does not count as a stable role anchor.

No external CSV artifacts are written; the notebook is the analysis artifact.


In [3]:
result = evaluate_models(
    MODEL_RUNS,
    INPUT_FOLDERS,
    anchor_min_minutes=ANCHOR_MIN_MINUTES,
    seed=SEED,
    control_samples_per_agent_day=CONTROL_SAMPLES_PER_AGENT_DAY,
)

representatives = select_representative_agents(
    result.agent_metrics,
    score_col="weekday_routine_continuity",
    n_per_model=1,
)

summary_cols = [
    "model",
    "n_agents",
    "n_agent_days",
    "worker_role_anchor_adherence_mean",
    "student_role_anchor_adherence_mean",
    "worker_anchor_start_std_median_minutes",
    "student_anchor_start_std_median_minutes",
    "weekday_routine_continuity_mean",
    "matched_control_similarity_mean",
    "weekday_routine_similarity_delta_mean",
]
summary = result.summary[summary_cols].copy()
summary.round(4)


,model,n_agents,n_agent_days,worker_role_anchor_adherence_mean,student_role_anchor_adherence_mean,worker_anchor_start_std_median_minutes,student_anchor_start_std_median_minutes,weekday_routine_continuity_mean,matched_control_similarity_mean,weekday_routine_similarity_delta_mean
0,OSS_20B,1000,7000,0.9977,0.9677,24.4949,26.8328,0.6536,0.5676,0.0860
1,OSS_120B,1000,7000,0.9996,0.9801,22.1923,33.4659,0.7426,0.6166,0.1260
2,OSS_20B_FT_SF,1000,6998,0.7848,0.7023,256.1231,210.7131,0.4933,0.4390,0.0543
3,OSS_20B_FT_US,1000,6999,0.8530,0.7478,215.1937,190.1841,0.5053,0.4471,0.0582


## Main Result Reading

A high routine-continuity score means the same agent's Monday-Friday compressed activity chains are similar. The delta is the more important cross-model value: it asks whether same-agent continuity exceeds a matched same-role, same-weekday different-agent control.

The prompted OSS-120B run is expected to be the strongest continuity baseline before ablations. The fine-tuned 20B runs can still perform well on population-level NHTS matching while being less individually routine-stable; these metrics intentionally measure a different behavioral dimension.


In [4]:
metric_rows = [
    {
        "metric": "role_anchor_adherence",
        "direction": "higher is stronger role continuity",
        "definition": METRIC_DEFINITIONS["role_anchor_adherence"],
    },
    {
        "metric": "anchor_start_time_stability",
        "direction": "lower is more stable",
        "definition": METRIC_DEFINITIONS["anchor_start_time_stability"],
    },
    {
        "metric": "weekday_routine_continuity",
        "direction": "higher is more self-similar across weekdays",
        "definition": METRIC_DEFINITIONS["weekday_routine_continuity"],
    },
    {
        "metric": "weekday_routine_similarity_delta",
        "direction": "higher means same-agent continuity exceeds matched controls",
        "definition": METRIC_DEFINITIONS["weekday_routine_similarity_delta"],
    },
]
pd.DataFrame(metric_rows)


,metric,direction,definition
0,role_anchor_adherence,higher is stronger role continuity,"For workers and students, the fraction of week..."
1,anchor_start_time_stability,lower is more stable,For workers and students with at least three o...
2,weekday_routine_continuity,higher is more self-similar across weekdays,"For each agent, the mean normalized edit-dista..."
3,weekday_routine_similarity_delta,higher means same-agent continuity exceeds mat...,Weekday routine continuity minus a matched con...


## Representative High/Low Samples

The table below selects the highest- and lowest-scoring agent per model according to `weekday_routine_continuity`. These are not cherry-picked claims; they are metric-facing inspection cases used to check whether the score rewards the kind of routine continuity we intend.


In [5]:
representatives.round(4)


,model,example_type,example_rank,agent_id,role_label,agent_type,weekday_routine_continuity,weekday_routine_similarity_delta,matched_control_similarity,role_anchor_adherence,anchor_start_time_stability
0,OSS_20B,high,1,8477,student,2,1.0000,0.3334,0.6666,1.0000,21.0357
1,OSS_20B,low,1,21011,homemaker,3,0.3373,-0.0902,0.4275,NaN,NaN
2,OSS_120B,high,1,4451,student,2,1.0000,0.1907,0.8093,1.0000,29.4534
3,OSS_120B,low,1,53684,worker,1,0.4889,0.0974,0.3915,1.0000,626.9709
4,OSS_20B_FT_SF,high,1,33113,student,2,1.0000,0.3687,0.6313,0.8000,14.3614
5,OSS_20B_FT_SF,low,1,23210,student,2,0.1667,-0.0567,0.2234,0.6667,NaN
6,OSS_20B_FT_US,high,1,25915,student,2,1.0000,0.4038,0.5962,1.0000,51.2835
7,OSS_20B_FT_US,low,1,18648,worker,1,0.1494,-0.1036,0.2530,0.6000,588.1326


## Chain-Level Inspection

For each representative agent, inspect the five weekday compressed chains. High-scoring examples should show similar weekday skeletons, while low-scoring examples should show large changes in the weekday chain structure, missing anchors, or irregular ordering.


In [6]:
for row in representatives.itertuples(index=False):
    display(Markdown(
        f"### {row.model} / {row.example_type.title()} continuity example: agent {int(row.agent_id)}"
    ))
    chain_table = get_agent_weekday_chains(
        result.agent_days,
        model=row.model,
        agent_id=int(row.agent_id),
    )
    display(chain_table)


### OSS_20B / High continuity example: agent 8477

,model,agent_id,date,day_name,role_label,activity_chain_text,role_anchor_present,role_anchor_start,role_anchor_minutes
707,OSS_20B,8477,2025-09-08,Monday,student,Home -> Education -> Home -> Recreational -> Home,1.0,475.0,365.0
708,OSS_20B,8477,2025-09-09,Tuesday,student,Home -> Education -> Home -> Recreational -> Home,1.0,475.0,365.0
709,OSS_20B,8477,2025-09-10,Wednesday,student,Home -> Education -> Home -> Recreational -> Home,1.0,490.0,365.0
710,OSS_20B,8477,2025-09-11,Thursday,student,Home -> Education -> Home -> Recreational -> Home,1.0,525.0,320.0
711,OSS_20B,8477,2025-09-12,Friday,student,Home -> Education -> Home -> Recreational -> Home,1.0,480.0,365.0


### OSS_20B / Low continuity example: agent 21011

,model,agent_id,date,day_name,role_label,activity_chain_text,role_anchor_present,role_anchor_start,role_anchor_minutes
1666,OSS_20B,21011,2025-09-08,Monday,homemaker,Home -> Recreational -> Shopping -> Education ...,NaN,NaN,NaN
1667,OSS_20B,21011,2025-09-09,Tuesday,homemaker,Home,NaN,NaN,NaN
1668,OSS_20B,21011,2025-09-10,Wednesday,homemaker,Home -> Education -> Home -> Recreational -> H...,NaN,NaN,NaN
1669,OSS_20B,21011,2025-09-11,Thursday,homemaker,Home -> Education -> Shopping -> Home -> Recre...,NaN,NaN,NaN
1670,OSS_20B,21011,2025-09-12,Friday,homemaker,Home -> Education -> Home -> Community -> Home...,NaN,NaN,NaN


### OSS_120B / High continuity example: agent 4451

...

,model,agent_id,date,day_name,role_label,activity_chain_text,role_anchor_present,role_anchor_start,role_anchor_minutes
15897,OSS_20B_FT_SF,23210,2025-09-08,Monday,student,Home,0.0,NaN,0.0
15898,OSS_20B_FT_SF,23210,2025-09-11,Thursday,student,Eat Meal -> Education,1.0,455.0,310.0
15899,OSS_20B_FT_SF,23210,2025-09-12,Friday,student,Other -> Education,1.0,535.0,405.0


### OSS_20B_FT_US / High continuity example: agent 25915

,model,agent_id,date,day_name,role_label,activity_chain_text,role_anchor_present,role_anchor_start,role_anchor_minutes
23125,OSS_20B_FT_US,25915,2025-09-08,Monday,student,Home -> Education -> Home,1.0,430.0,530.0
23126,OSS_20B_FT_US,25915,2025-09-09,Tuesday,student,Home -> Education -> Home,1.0,350.0,440.0
23127,OSS_20B_FT_US,25915,2025-09-10,Wednesday,student,Home -> Education -> Home,1.0,485.0,365.0
23128,OSS_20B_FT_US,25915,2025-09-11,Thursday,student,Home -> Education -> Home,1.0,440.0,295.0
23129,OSS_20B_FT_US,25915,2025-09-12,Friday,student,Home -> Education -> Home,1.0,390.0,520.0


### OSS_20B_FT_US / Low continuity example: agent 18648

,model,agent_id,date,day_name,role_label,activity_chain_text,role_anchor_present,role_anchor_start,role_anchor_minutes
22467,OSS_20B_FT_US,18648,2025-09-08,Monday,worker,Home -> Other -> Recreational -> Home -> Eat M...,1.0,1170.0,270.0
22468,OSS_20B_FT_US,18648,2025-09-09,Tuesday,worker,Work -> Eat Meal -> Work -> Recreational,1.0,0.0,1350.0
22469,OSS_20B_FT_US,18648,2025-09-10,Wednesday,worker,Home -> Recreational -> Other -> Eat Meal,0.0,NaN,0.0
22470,OSS_20B_FT_US,18648,2025-09-11,Thursday,worker,Eat Meal -> Home,0.0,NaN,0.0
22471,OSS_20B_FT_US,18648,2025-09-12,Friday,worker,Home -> Care -> Shopping -> Other -> Care -> S...,1.0,690.0,135.0


## Story Inspection From Logs

The rendered stories below use the existing prompt/decision/reflection logs. They hide reasoning text to keep the inspection focused on observable plans, step activities, rationales, and reflections. Use these examples to check whether the metric scores correspond to the qualitative traces:

- **High examples** should have repeated weekday anchors and similar activity-chain skeletons.
- **Low examples** should visibly drift across weekdays, miss role anchors, or change chain structure enough that a low continuity score is plausible.


In [7]:
STORY_DAYS = [1, 2, 3, 4, 5]

for row in representatives.itertuples(index=False):
    display(Markdown(
        f"### Story: {row.model} / {row.example_type.title()} continuity example / agent {int(row.agent_id)}"
    ))
    _ = show_agent_story_compare(
        agent_id=int(row.agent_id),
        sim_runs=MODEL_RUNS,
        run_input_folders=INPUT_FOLDERS,
        models=[row.model],
        days=STORY_DAYS,
        show_day_plan_reasoning=False,
        show_step_reasoning=False,
        show_reflection_reasoning=False,
        show_reflection=True,
    )


### Story: OSS_20B / High continuity example / agent 8477

Time,Activity,Rationale,Stay (min)
07:05,Home (1),"I’m still at home, grabbing breakfast before heading to school at 7:30.",25
07:35,Home (1),I’m just finishing my breakfast at home; a short 10‑minute stretch will let me eat and get ready before heading to school.,10
07:50,Education (4),I’m still at home but the school day starts at 8 a.m.; I need to sprint out and catch the first class.,360
14:00,Home (1),"School ends at 14:00, so I’ll head home to finish any leftover homework and get ready for the walk with my sister later.",120
16:10,Home (1),"Still have short assignments to finish outside of class, so I will stay home a bit longer before heading out for a walk with my sister.",60
17:15,Home (1),I still have a few assignments left from school that I need to finish before going out with my sister.,30
17:50,Home (1),"I still have a few assignments to finish before heading out with my sister for a walk, so I’ll stay at home for a bit longer.",30
18:25,Home (1),"I still have a few assignments left from school, so I'll stay home for a while longer to finish them before heading out for the walk with my sister.",15
18:45,Home (1),"I still have a handful of assignments to finish before walking with my sister, so I’ll stay at home for a short while longer.",30
19:20,Recreational (5),Heading out for a quick walk with my sister to stretch before we head home for dinner,30


### Story: OSS_20B / Low continuity example / agent 21011

Time,Activity,Rationale,Stay (min)
07:05,Home (1),"Begin the day’s routine at home—help the under‑6 and 6‑to‑17 year olds dress, eat breakfast, and get them ready for school hand‑over at 8:00 a.m.",55
08:05,Recreational (5),"Kids have already been handed over to school at 8:00, so I’ll head out for a short walk in the nearby park to begin the day with some exercise and fresh air.",30
09:05,Shopping (6),"I finished the brief walk in the park. Time to run errands: drive to the nearby grocery store, pick up any medications I need, and replenish household supplies.",45
10:05,Shopping (6),"I still need to pick up a prescription from the pharmacy in the same grocery store, so I'll stay a bit longer to finish my shopping list and get my medication before heading home.",10
10:20,Shopping (6),I’m almost done in the store—just picking up my prescription and last few grocery items—so I’ll finish the errands in about ten more minutes before heading to the kids’ afternoon activity.,10
10:35,Education (4),"I’m still at the store, so I’ll finish any remaining grocery/medication items quickly, then drive to drop the children from their after‑school activity. A brief 5‑minute stop after traffic will let me hand them off and get home for dinner prep.",5
11:05,Education (4),I’m at the after‑school program today; I’ll wait for the children to arrive before dropping them off at the start of the afternoon activity and then drive home for dinner prep.,55
12:05,Education (4),"Having just dropped the kids off at their afternoon activity, I’ll stay a short while longer to ensure they settle in and finish any small check‑ins before heading home for dinner preparation.",15
12:25,Education (4),Hold a short moment at the afternoon activity to ensure the children settle in before I head home to finish dinner prep.,10
12:40,Education (4),Staying a little longer to let the children finish settling in and make a quick last check‑in before heading home to finish dinner preparation.,20


### Story: OSS_120B / High continuity example / agent 4451

...

Time,Activity,Rationale,Stay (min)
07:05,Education (4),"Heading to high school for the day, attending classes and after-school activities.",585
07:10,Education (4),"I'm at school for the whole day. I'll be attending my classes, hanging out with friends during breaks, and finishing up my school work until late afternoon before heading home.",740
19:35,Home (1),"School is finally over for the day. I'm heading straight home to finish up my homework, help my family with the evening, and then relax and sleep for the rest of the night.",305
19:40,Home (1),"I'm running a little behind schedule after school, so I'm going to stay home for the rest of the evening to finish my homework, help out around the house, and get some sleep before tomorrow.",260
Time,Activity,Rationale,Stay (min)
00:05,Home (1),"It's the night, so I'm going to sleep for about six and a half hours before waking up to get ready for school.",385
06:35,Education (4),It's time to head to high school for my classes today. I'm leaving early to catch my usual bus and get settled before the day officially begins.,545
06:40,Education (4),I'm really early just to grab my locker before any classes officialy start.,5
06:50,Education (4),"I'm almost an hour early for school. I can't be wet on the campus lighting walk, so I need to grab a dry towel from a locker to dry up before taking the bus home and potentially running back to school when my day ends.",23
07:20,Education (4),"I finished up with the lockers and my sibling, so now it's time to head to the front of the school to catch the bus to campus. I'll be at school for the rest of the day until my classes finish at 4:00 PM.",520


### Story: OSS_20B_FT_US / High continuity example / agent 25915

Time,Activity,Rationale,Stay (min)
07:05,Education (4),It's time to head to school for the day. I'll be in classes and playing with my friends until the afternoon.,525
16:00,Home (1),"School is finally over for the day! I'm heading home to play, do my homework, have dinner with my family, and relax for the rest of the evening.",480
Time,Activity,Rationale,Stay (min)
00:10,Home (1),I'm still asleep and need to get ready in the morning. I'll stay in bed for a little bit longer to rest up before I head out for school.,30
00:45,Home (1),It's still very early and I'm very tired. I'll sleep in bed for a few more hours to get some rest before I need to eat breakfast and get ready for school tomorrow.,295
05:45,Education (4),It's getting close to sunrise and my breakfast is getting ready to eat. I need to head to school for a full day of classes. I'll be there for most of the day before coming back home.,435
13:10,Home (1),"School is over for the day. I'm going to head straight back home to do my homework, spend time with my family, and eventually go to sleep.",530
22:10,Home (1),"It's getting very late, and I'm exhausted from school and doing my homework all afternoon. I'm going to stay home, maybe read a book or talk to my family briefly, and then go to sleep for the night.",110
Time,Activity,Rationale,Stay (min)
00:05,Home (1),"It's just past midnight, so I'm going to sleep and get ready for school tomorrow morning. I'll be at home for the night until my parents get up and we have breakfast.",470


### Story: OSS_20B_FT_US / Low continuity example / agent 18648

Time,Activity,Rationale,Stay (min)
07:05,Other (9),"Running a quick morning errand, like dropping off some mail or picking up gasoline, before my partner and I head out for our workout.",5
07:40,Recreational (5),"Now that my errand is done, my partner and I are heading out for our morning workout session before lunch.",80
09:15,Home (1),Wrapping up our workout and taking a quick break with my partner to check something out at home before we head for lunch.,5
09:45,Home (1),"After our morning workout, I'm going to take a bit of time back at the house to browse my medical home app and organize the latest doctor’s notes before my afternoon appointment.",15
10:05,Eat Meal (3),"Now that I've sorted through my medical notes, it's time for my partner and me to head out and grab some lunch. After eating, we plan to take a relaxing afternoon walk in a nearby neighborhood park to enjoy the day.",165
13:20,Recreational (5),"That was a good, long lunch. Before I have to head to my health checkup pretty soon, my partner and I are going to take an extended walk around a local neighborhood spot to soak in the afternoon light and enjoy the day.",85
15:05,Care (7),"We've had a nice long walk, but now it's time to head over to the clinic for my long afternoon health checkup. It should take a while to go through all the tests and talk with the doctor.",110
17:10,Other (9),"My health appointment is finally done, though it took much longer than I expected. I'm still at the clinic to pick up my prescriptions and grab a couple of quick things before heading back to my partner.",45
18:15,Home (1),I've finally finished getting my prescriptions and the extra stuff I needed from the clinic. Now I'm going to make a quick stop at home to drop things off and check on my partner before we both head back out to grab dinner.,21
19:00,Home (1),"We're grabbing dinner down there and are heading back. Before the main shift at the center, I need to sit down to check in with my partner for a few minutes to settle in.",10


## Notebook Outputs

This notebook intentionally does not write CSV or other side artifacts. The summary table, representative-agent table, chain inspection, and rendered stories are kept as executed notebook outputs.


In [8]:
{
    "models_evaluated": list(MODEL_RUNS.keys()),
    "agent_metric_rows": int(result.agent_metrics.shape[0]),
    "agent_day_rows": int(result.agent_days.shape[0]),
    "representative_examples": int(representatives.shape[0]),
}


{'models_evaluated': ['OSS_20B', 'OSS_120B', 'OSS_20B_FT_SF', 'OSS_20B_FT_US'],
 'agent_metric_rows': 4000,
 'agent_day_rows': 27997,
 'representative_examples': 8}

## Notes

These results complete metric definition and validation on existing outputs. A separate ablation notebook can reuse the same metric functions without changing the definitions.
